# Combined NOAA Weather Visualization Project

This notebook combines **jaswinder_s1, jaswinder_s2, jaswinder_s3, jaswinder_s4, and jaswinder_s5** in order.

All original cells from the five notebooks are preserved.

# =============================
# jaswinder_s1
# =============================

# jaswinder_s1

## Collect Initial Dataset and Inspect Raw Data

This notebook is designed for the first stage of the project.

### Objectives
- collect the initial dataset through one of these methods:
  - API
  - CSV upload
  - scraping
  - database usage
- inspect dataset structure and variables
- prepare raw data for GitHub upload
- document citations properly

### Selected workflow for this project
For this submission, the dataset is collected through **CSV upload / direct CSV loading** using the NOAA weather dataset for Newark, New Jersey.

### Citation
National Oceanic and Atmospheric Administration (NOAA), National Centers for Environmental Information. Global Historical Climatology Network Daily data for station USW00014734.


## Step 1: Import libraries

In [ ]:
import pandas as pd
import requests
import sqlite3
from bs4 import BeautifulSoup

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


## Step 2: Raw data source

This project uses a raw NOAA CSV file.

**Source citation:**  
NOAA National Centers for Environmental Information. Global Historical Climatology Network Daily dataset, station USW00014734.

**Direct source used in this notebook:**  
https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/access/USW00014734.csv


In [ ]:
csv_url = "https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/access/USW00014734.csv"
print(csv_url)


## Step 3: Collect initial dataset by CSV loading

This is the main collection method used for the project.


In [ ]:
df_raw = pd.read_csv(csv_url)
df_raw.head()


## Step 4: Inspect dataset structure

In [ ]:
print("Shape of raw dataset:", df_raw.shape)
print("\nColumn names:")
print(df_raw.columns.tolist())


In [ ]:
df_raw.info()


In [ ]:
df_raw.describe(include='all')


## Step 5: Inspect variables

Important variables for this weather project usually include:
- DATE
- PRCP
- SNOW
- SNWD
- TMAX
- TMIN
- AWND

This step checks whether these variables exist in the raw data.


In [ ]:
important_vars = ['DATE', 'PRCP', 'SNOW', 'SNWD', 'TMAX', 'TMIN', 'AWND']
available_vars = [c for c in important_vars if c in df_raw.columns]
missing_vars = [c for c in important_vars if c not in df_raw.columns]

print("Available variables:", available_vars)
print("Missing variables:", missing_vars)

df_raw[available_vars].head()


## Step 6: Inspect missing values

In [ ]:
missing_values = df_raw.isnull().sum().sort_values(ascending=False)
missing_values[missing_values > 0]


## Step 7: Inspect date coverage

In [ ]:
if 'DATE' in df_raw.columns:
    df_raw['DATE'] = pd.to_datetime(df_raw['DATE'], errors='coerce')
    print("Start date:", df_raw['DATE'].min())
    print("End date:", df_raw['DATE'].max())
    print("Total observations:", len(df_raw))
else:
    print("DATE column not found.")


## Step 8: Alternative collection methods

The assignment mentions API, CSV upload, scraping, and database usage.  
The project uses CSV loading as the main method, but the notebook also documents the other methods.


### 8A. API example template

In [ ]:
# Example template only
# api_url = "https://api.example.com/data"
# response = requests.get(api_url)
# if response.status_code == 200:
#     api_data = response.json()
#     df_api = pd.DataFrame(api_data)
#     display(df_api.head())
# else:
#     print("API request failed")


### 8B. CSV upload example for Google Colab

In [ ]:
# Example template only
# from google.colab import files
# import io
# uploaded = files.upload()
# file_name = list(uploaded.keys())[0]
# df_uploaded = pd.read_csv(io.BytesIO(uploaded[file_name]))
# df_uploaded.head()


### 8C. Scraping example template

In [ ]:
# Example template only
# page_url = "https://example.com/table-page"
# response = requests.get(page_url)
# if response.status_code == 200:
#     tables = pd.read_html(response.text)
#     if tables:
#         df_scraped = tables[0]
#         display(df_scraped.head())
#     else:
#         print("No HTML table found")
# else:
#     print("Webpage request failed")


### 8D. Database usage example

This step stores the raw dataset in SQLite and reads a sample back.


In [ ]:
conn = sqlite3.connect("raw_weather.db")
df_raw.to_sql("noaa_raw_weather", conn, if_exists="replace", index=False)

sample_from_db = pd.read_sql_query("SELECT * FROM noaa_raw_weather LIMIT 10;", conn)
conn.close()

sample_from_db


## Step 9: Save raw data locally for GitHub upload

In [ ]:
df_raw.to_csv("raw_noaa_weather.csv", index=False)
print("Saved raw_noaa_weather.csv")


## Step 10: GitHub commit instructions

This notebook cannot commit to GitHub automatically unless Git is configured in your Colab session.  
So for the assignment, use these steps:

1. Download `raw_noaa_weather.csv` from Colab.
2. Upload it to your GitHub repository.
3. Add a `README.md` file with the citation below.
4. Commit with a message like:  
   `Add raw NOAA weather dataset with source citation`

### Suggested README citation text
Source: National Oceanic and Atmospheric Administration (NOAA), National Centers for Environmental Information, Global Historical Climatology Network Daily dataset, station USW00014734. Accessed from NOAA public climate data.

### Suggested repository files
- `raw_noaa_weather.csv`
- `README.md`
- `jaswinder_s1.ipynb`


## Step 11: Short submission note

The initial dataset was collected using direct CSV loading from the NOAA climate data source. The raw file was inspected to identify the structure, variable names, data types, date coverage, and missing values. The dataset was then saved in raw form for version control and GitHub upload, with source citation included for transparency and reproducibility.


# =============================
# jaswinder_s2
# =============================

# jaswinder_s2

## Data Cleaning, Organization and Exploratory Data Analysis (EDA)

### Objectives
- Clean and organize NOAA dataset using Pandas
- Handle:
  - Missing values
  - Inconsistent values
  - Outliers
- Create at least 3 visualizations
- Provide short written observations


## Step 1: Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)


## Step 2: Load Raw Dataset

In [ ]:
url = "https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/access/USW00014734.csv"
df = pd.read_csv(url)
df.head()


## Step 3: Select Relevant Variables

In [ ]:
cols = ['DATE','PRCP','TMAX','TMIN','AWND']
df = df[cols]
df.head()


## Step 4: Data Type Conversion

In [ ]:
df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')

# Convert NOAA units
df['TMAX'] = df['TMAX'] / 10
df['TMIN'] = df['TMIN'] / 10
df['PRCP'] = df['PRCP'] / 10


## Step 5: Handle Missing Values

In [ ]:
print(df.isnull().sum())

df['PRCP'] = df['PRCP'].fillna(0)
df['TMAX'] = df['TMAX'].fillna(df['TMAX'].mean())
df['TMIN'] = df['TMIN'].fillna(df['TMIN'].mean())
df['AWND'] = df['AWND'].fillna(df['AWND'].mean())


## Step 6: Handle Inconsistent Data

In [ ]:
# Ensure no negative precipitation
df = df[df['PRCP'] >= 0]


## Step 7: Handle Outliers

In [ ]:
df = df[(df['TMAX'] < 60) & (df['TMIN'] > -40)]


## Step 8: Create New Features

In [ ]:
df['Month'] = df['DATE'].dt.month
df['Year'] = df['DATE'].dt.year


## Step 9: Visualization 1 - Temperature Trend

In [ ]:
plt.figure()
df.groupby('DATE')['TMAX'].mean().plot()
plt.title("Temperature Trend")
plt.xlabel("Date")
plt.ylabel("Max Temperature")
plt.show()


**Observation:** Temperature follows a seasonal pattern with clear peaks and drops.

## Step 10: Visualization 2 - Monthly Rainfall

In [ ]:
df.groupby('Month')['PRCP'].sum().plot(kind='bar')
plt.title("Monthly Rainfall")
plt.xlabel("Month")
plt.ylabel("Rainfall")
plt.show()


**Observation:** Rainfall varies significantly across months, indicating seasonal variation.

## Step 11: Visualization 3 - Temperature Distribution

In [ ]:
df['TMAX'].hist()
plt.title("Temperature Distribution")
plt.xlabel("Temperature")
plt.ylabel("Frequency")
plt.show()


**Observation:** Most temperature values are concentrated in a moderate range with fewer extremes.

## Step 12: Save Clean Dataset

In [ ]:
df.to_csv("clean_noaa_weather.csv", index=False)
print("Dataset saved")


## Final Notes

### Work Completed
- Cleaned dataset
- Handled missing values
- Removed inconsistencies and outliers
- Created 3 visualizations

### Conclusion
The dataset is now organized and ready for further analysis and visualization.


# =============================
# jaswinder_s3
# =============================

# jaswinder_s3

## Visual Exploration Checkpoint

### Objectives
- Explore multiple chart types using the same dataset
- Experiment with grouping, aggregation, and scale
- Identify effective and misleading visuals
- Provide evaluation and reasoning

Dataset: Clean NOAA Weather Data


## Step 1: Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("clean_noaa_weather.csv")
df['DATE'] = pd.to_datetime(df['DATE'])
df['Month'] = df['DATE'].dt.month
df['Year'] = df['DATE'].dt.year
df.head()


## Visual 1: Line Chart (Daily Temperature Trend)

In [ ]:
plt.figure()
df.groupby('DATE')['TMAX'].mean().plot()
plt.title("Daily Temperature Trend")
plt.show()


**Evaluation:** Effective. Shows continuous trend clearly.

## Visual 2: Aggregated Line Chart (Monthly Avg Temp)

In [ ]:
df.groupby('Month')['TMAX'].mean().plot()
plt.title("Monthly Avg Temperature")
plt.show()


**Evaluation:** Effective. Aggregation improves clarity.

## Visual 3: Bar Chart (Monthly Rainfall)

In [ ]:
df.groupby('Month')['PRCP'].sum().plot(kind='bar')
plt.title("Monthly Rainfall")
plt.show()


**Evaluation:** Strong. Best for comparison.

## Visual 4: Scatter Plot (Temp vs Wind)

In [ ]:
plt.scatter(df['TMAX'], df['AWND'])
plt.title("Temp vs Wind")
plt.show()


**Evaluation:** Weak. No clear relationship visible. ❌ DISCARD

## Visual 5: Pie Chart (Rainfall Distribution)

In [ ]:
df.groupby('Month')['PRCP'].sum().plot(kind='pie')
plt.title("Rainfall Distribution")
plt.show()


**Evaluation:** Misleading. Too many categories. ❌ DISCARD

## Visual 6: Box Plot (Temperature by Month)

In [ ]:
df.boxplot(column='TMAX', by='Month')
plt.title("Temp Variability by Month")
plt.suptitle("")
plt.show()


**Evaluation:** Very effective. Shows spread and outliers clearly.

## Final Visual Exploration Summary

### Effective Visuals
- Line chart (daily and monthly)
- Bar chart (rainfall comparison)
- Box plot (variability)

### Discarded Visuals
- Scatter plot → unclear relationship
- Pie chart → difficult comparison

### Conclusion
Different visualizations reveal different insights. Aggregation improves clarity, while poor chart selection can mislead interpretation.


# =============================
# jaswinder_s4
# =============================

# jaswinder_s4

## Assumptions, Revised Visuals, and Interpretation Risks

### Objectives
- Identify assumptions behind visuals
- Examine:
  - Data completeness
  - Representativeness
  - Variable definitions
- Revise visuals to:
  - Show variability
  - Avoid misleading encodings
- Provide final written reflection


## Step 1: Import and Load Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("clean_noaa_weather.csv")
df['DATE'] = pd.to_datetime(df['DATE'])
df['Month'] = df['DATE'].dt.month
df['Year'] = df['DATE'].dt.year

df.head()


## Step 2: Assumptions - Data Completeness

Check if dataset has missing values or gaps.


In [ ]:
print(df.isnull().sum())
print("Date range:", df['DATE'].min(), "to", df['DATE'].max())


### Interpretation
The dataset may have missing days or values. A viewer may assume continuous data even if gaps exist.


## Step 3: Assumptions - Representativeness

This dataset comes from one location (Newark). It does not represent:
- entire New Jersey
- entire USA
- global climate patterns


### Interpretation
Viewers may incorrectly generalize results beyond this location.


## Step 4: Assumptions - Variable Definitions

Variables:
- TMAX = daily maximum temperature (°C)
- TMIN = daily minimum temperature (°C)
- PRCP = precipitation (mm)

Misunderstanding units can lead to wrong interpretation.


## Step 5: Misleading Visual (Before Revision)

In [ ]:
df.groupby('Month')['PRCP'].sum().plot(kind='pie')
plt.title("Rainfall Distribution (Misleading)")
plt.show()


### Problem
Pie charts make comparison difficult when many categories exist.


## Step 6: Revised Visual - Bar Chart

In [ ]:
df.groupby('Month')['PRCP'].sum().plot(kind='bar')
plt.title("Monthly Rainfall (Improved)")
plt.xlabel("Month")
plt.ylabel("Rainfall")
plt.show()


### Improvement
Bar chart improves comparison and readability.


## Step 7: Revised Visual - Variability (Box Plot)

In [ ]:
df.boxplot(column='TMAX', by='Month')
plt.title("Temperature Variability")
plt.suptitle("")
plt.show()


### Improvement
Box plot shows spread, median, and outliers.


## Step 8: Revised Visual - Dual Line Chart

In [ ]:
monthly = df.groupby('Month')[['TMAX','TMIN']].mean()

monthly.plot()
plt.title("Avg Monthly Max and Min Temperature")
plt.xlabel("Month")
plt.ylabel("Temperature")
plt.show()


### Improvement
Shows relationship between high and low temperatures.


## Step 9: Final Written Submission

### What could someone misunderstand from our visuals?

Someone could misunderstand our visuals by assuming that the dataset is complete and fully representative of broader weather conditions. Since the data is collected from a single location, it may not reflect regional or national patterns. Additionally, using averages can hide variability, leading viewers to believe that temperature and rainfall are more stable than they actually are.

Another issue arises from the choice of visualization. For example, pie charts can distort comparisons between months, making small differences appear significant or unclear. Without clear labels and explanations of variables such as temperature and precipitation units, viewers may misinterpret the data. Revised visualizations that show variability and use clearer chart types help reduce these misunderstandings.


## Final Takeaway

Good visualizations require:
- correct assumptions
- proper chart selection
- clear representation of variability
- awareness of audience interpretation


# =============================
# jaswinder_s5
# =============================

# jaswinder_s5

## Partial Visualization Report – Re-examining Findings

### Objectives
- Re-examine earlier findings
- Identify alternative interpretations
- Improve or replace misleading visuals
- Provide updated interpretations
- Explain what changed and why


## Step 1: Load Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("clean_noaa_weather.csv")
df['DATE'] = pd.to_datetime(df['DATE'])
df['Month'] = df['DATE'].dt.month
df['Year'] = df['DATE'].dt.year

df.head()


## Step 2: Original Finding (Temperature Trend)

Earlier interpretation:
Temperature follows a seasonal pattern.


In [ ]:
plt.figure()
df.groupby('DATE')['TMAX'].mean().plot()
plt.title("Original: Temperature Trend")
plt.show()


### Alternative Interpretation
The trend may appear smooth, but it hides daily fluctuations and extreme values.


## Step 3: Improved Visualization (Rolling Average + Raw Data)

In [ ]:
df_sorted = df.sort_values('DATE')
df_sorted['Rolling'] = df_sorted['TMAX'].rolling(window=7).mean()

plt.figure()
plt.plot(df_sorted['DATE'], df_sorted['TMAX'], alpha=0.3, label='Daily Temp')
plt.plot(df_sorted['DATE'], df_sorted['Rolling'], label='7-day Avg')
plt.legend()
plt.title("Improved: Trend with Variability")
plt.show()


### Why this is better
Shows both variability and trend instead of oversimplifying.


## Step 4: Original Rainfall Visualization (Bar Chart)

In [ ]:
monthly_rain = df.groupby('Month')['PRCP'].sum()

monthly_rain.plot(kind='bar')
plt.title("Original: Monthly Rainfall")
plt.show()


### Alternative Interpretation
Total rainfall may exaggerate differences if some months have more days recorded.


## Step 5: Improved Visualization (Average Rainfall per Day)

In [ ]:
avg_rain = df.groupby('Month')['PRCP'].mean()

avg_rain.plot(kind='bar')
plt.title("Improved: Avg Daily Rainfall")
plt.show()


### Why this is better
Normalizes rainfall, making months comparable.


## Step 6: Original Distribution (Histogram)

In [ ]:
df['TMAX'].hist()
plt.title("Original: Temperature Distribution")
plt.show()


### Issue
Histogram alone hides seasonal grouping.


## Step 7: Improved Visualization (Box Plot by Month)

In [ ]:
df.boxplot(column='TMAX', by='Month')
plt.title("Improved: Monthly Distribution")
plt.suptitle("")
plt.show()


### Why this is better
Shows spread, outliers, and seasonal variation.


## Step 8: Final Written Report

### Updated Findings
The initial analysis suggested clear seasonal patterns in temperature and rainfall. However, further examination shows that these patterns are more complex than initially presented. Daily variability, extreme values, and differences in aggregation significantly affect interpretation.

### What Changed
Earlier visuals relied on averages and totals, which simplified the data. The revised visuals include rolling averages, normalized rainfall, and distribution plots to better represent variability.

### Why It Changed
The updated visuals provide a more accurate representation of the dataset by avoiding oversimplification and highlighting variation within the data. This reduces the risk of misleading conclusions and improves interpretability.

### Conclusion
Visualization choices strongly influence interpretation. More detailed and context-aware visuals lead to better insights.
